# Reproducing Logic-LM (Pan et al., Findings of EMNLP 2023)

Runs the authors' **symbolic-solver stage** on the logic programs **they** generated with GPT-4
(`outputs/logic_programs`), with 3 seeds. No OpenAI key and no GPU needed (`Runtime > Change runtime type > CPU` is fine).

Run the cells top to bottom. Every cell writes its full output to `logs/`.
**If a cell fails, do not delete it.** Write the error and your fix in `NOTES.md` (Attempt 1, 2, 3...) and re-run.

In [ ]:
# 1. Your fork (change if your username differs)
GITHUB_USER = "suriyakala-gif"
REPO = "Logic-LLM"
!git clone https://github.com/{GITHUB_USER}/{REPO}.git
%cd {REPO}
!mkdir -p logs results

In [ ]:
# 2. Record the environment for SETUP.md
!( date -u; echo; python --version; echo; cat /etc/os-release | head -2; echo; nproc; free -h | head -2; echo; nvidia-smi -L 2>/dev/null || echo "GPU: none (CPU runtime)" ) 2>&1 | tee logs/00_environment.log

In [ ]:
# 3. Install exactly as the README says. If this fails -> NOTES.md Attempt 1, then try the next cell.
!pip install -r requirements.txt 2>&1 | tee logs/01_install.log

In [ ]:
# 3b. ONLY if 3 failed: install the solver packages without the old pins, and log it.
# !pip install z3-solver python-constraint nltk tqdm 2>&1 | tee logs/01b_install_fallback.log
# !pip install pyke3 2>&1 | tee -a logs/01b_install_fallback.log   # Pyke for ProntoQA/ProofWriter

In [ ]:
# 4. Prover9 (needed for FOLIO). The repo ships a Linux binary; make sure it is executable.
!find . -path "*Prover9*" -name "prover9" -type f 2>/dev/null | tee logs/02_prover9.log
!find . -path "*Prover9*bin*" -type f -exec chmod +x {} \; 2>&1 | tee -a logs/02_prover9.log
# If FOLIO later fails with a Prover9 error, try the system package and note it in NOTES.md:
# !apt-get -y install prover9 2>&1 | tee -a logs/02_prover9.log

In [ ]:
# 5. Which saved logic programs exist? (This decides which datasets you can reproduce.)
!ls -la outputs/logic_programs/ 2>&1 | tee logs/03_available_programs.log

In [ ]:
# 6. One run EXACTLY as the README shows, to prove the authors' command works as written.
!python models/logic_inference.py \
    --model_name gpt-4 --dataset_name FOLIO --split dev \
    --backup_strategy random \
    --backup_LLM_result_path ./baselines/results/CoT_FOLIO_dev_gpt-4.json 2>&1 | tee logs/04_readme_command_FOLIO.log
!python models/evaluation.py --dataset_name FOLIO --model_name gpt-4 --split dev --backup random 2>&1 | tee -a logs/04_readme_command_FOLIO.log

In [ ]:
# 7. Full reproduction: 5 datasets x 3 seeds -> results/my_results.csv + logs/inference_*.log
!python run_reproduction.py --seeds 0 1 2 --split dev --model gpt-4 2>&1 | tee logs/05_reproduction.log
!cat results/my_results.csv

### Before the next cell
Open `results/paper_values.csv` (Files panel, left) and type in the paper's **Logic-LM (GPT-4)** accuracy for each dataset,
copied from the main results table in the paper PDF. Save the file.

In [ ]:
# 8. Comparison table + figure
!python make_table.py 2>&1 | tee logs/06_table.log
!python make_figure.py --gray 2>&1 | tee logs/07_figure.log
from IPython.display import Image, display
display(Image("fig1_gray.png"))   # grayscale check (Step 3)

In [ ]:
# 9. Freeze package versions (Step 1) - overwrites the authors' requirements.txt; the original stays in git history.
!cp requirements.txt requirements_original.txt
!pip freeze > requirements.txt

In [ ]:
# 10. Download everything you need to commit to GitHub
!zip -r reproduction_outputs.zip logs results fig1.pdf fig1_gray.png requirements.txt requirements_original.txt
from google.colab import files
files.download("reproduction_outputs.zip")